In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

In [4]:
api_wrapper_arxiv = ArxivAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=500
)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

In [5]:
arxiv.invoke("Attention iss all your need")

'Published: 2011-04-19\nTitle: The Coronal Physics Investigator (CPI) Experiment for ISS: A New Vision for Understanding Solar Wind Acceleration\nAuthors: J. L. Kohl, S. R. Cranmer, J. C. Raymond, T. J. Norton, P. J. Cucchiaro, D. B. Reisenfeld, P. H. Janzen, B. D. G. Chandran, T. G. Forbes, P. A. Isenberg, A. V. Panasyuk, A. A. van Ballegooijen\nSummary: In February 2011 we proposed a NASA Explorer Mission of Opportunity program to develop and operate a large-aperture ultraviolet coronagraph spectr'

In [7]:
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [11]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [12]:
#tavily search tool
from langchain_community.tools.tavily_search import TavilySearchResults

tavily=TavilySearchResults()

C:\Users\HP\AppData\Local\Temp\ipykernel_20348\2578923781.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily=TavilySearchResults()


In [13]:
tavily.invoke("provide me the current ai news")

[{'title': 'AI News Briefs BULLETIN BOARD for April\xa02026 | Radical Data Science',
  'url': 'https://radicaldatascience.wordpress.com/2026/04/15/ai-news-briefs-bulletin-board-for-april-2026/',
  'content': '[4/8/2026] My picture of the present in AI – Ryan Greenblatt is the chief scientist at Redwood Research, a research organization with the mission of aligning superhuman AI. This post goes through some of his best guesses for the current situation of AI. The scenario forecast discusses R&D access regulations, engineering capabilities and qualitative abilities, misalignment and misalignment-related properties, cyber, bioweapons, and economic effects. Some of the claims are highly speculative, while others are better grounded. [...] Welcome to the AI News Briefs Bulletin Board, a timely new channel bringing you the latest industry insights and perspectives surrounding the field of AI including deep learning, large language models, generative AI, and transformers. I am working tireles

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [14]:
#combine all the tools
tools=[arxiv,wiki, tavily]

In [15]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
llm_with_tools=llm.bind_tools(tools)

In [17]:
from pprint import pprint
from langchain_core.messages import AIMessage,HumanMessage
llm_with_tools.invoke([HumanMessage(content="what is the recent ai news")])

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '17hrfa43m', 'function': {'arguments': '{"query":"recent AI news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 553, 'total_tokens': 573, 'completion_time': 0.02738323, 'completion_tokens_details': None, 'prompt_time': 0.051874256, 'prompt_tokens_details': None, 'queue_time': 0.051821092, 'total_time': 0.079257486}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019da520-0837-7541-88ed-866e1dd61ecc-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'recent AI news'}, 'id': '17hrfa43m', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 553, 'output_tokens': 20, 'total_tokens': 573})

In [18]:
from IPython.display import Image, display
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [19]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

In [20]:
#node definition
def tool_calling_llm(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

tools = [arxiv_tool, wikipedia_tool]

👉 Har tool ka:

name hota hai (e.g. "arxiv")
description hoti hai
🟢 Step 2: LLM ko tools ke saath bind karte ho
llm = llm.bind_tools(tools)

👉 Ab LLM ko pata hai:

“mere paas ye tools available hain”

🟢 Step 3: User input aata hai
"Attention is all you need"
🟢 Step 4: LLM decide karta hai

LLM internally sochta hai:

"Ye research paper lag raha hai → arxiv use karo"

👉 Output deta hai (hidden structured form me):

{
  "tool_calls": [
    {
      "name": "arxiv",
      "args": {"query": "Attention is all you need"}
    }
  ]
}
⚙️ Ab tumhara graph kya karta hai?
👇 Yeh line important hai
builder.add_conditional_edges("llm_tool", tools_condition)

👉 tools_condition check karta hai:

agar tool_calls hai → "tools" node pe jao
nahi hai → END
🔧 Tools node kya karta hai?
builder.add_node("tools", ToolNode(tools))

👉 Ye automatically:

LLM ka output dekhta hai
"name": "arxiv" padhta hai
tools list me dhundta hai
correct tool call karta hai
🔥 Simple Example

Tumhare tools:

tools = [arxiv, wikipedia]

LLM bola:

"name": "wikipedia"

👉 ToolNode karega:

wikipedia.invoke(...)
🧠 Ek line me samjho

LLM choose karta hai → ToolNode execute karta hai

In [22]:
builder=StateGraph(State)
builder.add_node("llm_tool",tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

#Add edge
builder.add_edge(START,"llm_tool")
builder.add_conditional_edges("llm_tool",tools_condition)
builder.add_edge("tools",END)

graph=builder.compile()